In [2]:
"""Train the three models on the Feb 2025 storm and check them on Sep 2024.

  1. link damaged   logistic regression, one for zones and one for transformers;
                    home risk = 1 - prod(1 - p_link) over the links on the home's path
  2. damage counts  Poisson regression per damage type, per zone
  3. access issue   logistic regression, per damage report

Reads the feature files from ~/aep/AEPHack/data/features/ (copied from s3://.../features/)
and the outage labels from S3. Writes models.joblib + metrics.json to ~/aep/AEPHack/data/models/.

Run in the SageMaker terminal:  pip install s3fs  &&  python train_models.py
"""
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

TRAIN, TEST = "feb2025", "sep2024"
DATA = os.path.expanduser(os.environ.get("AEP_DATA", "~/aep/AEPHack/data"))
FEATURES, MODELS = f"{DATA}/features/", f"{DATA}/models/"
LABELS = "s3://sagemaker-aep-outage-indiansassemble/processed/batch2/outage_labels.parquet"
DAMAGE_TYPES = ["pole", "primary_wire", "transformer", "device", "secondary"]

ZONE_X = [
    "customers", "is_recloser", "breaker_only", "primary_overhead_km", "primary_underground_km",
    "secondary_overhead_km", "secondary_underground_km", "customers_per_km", "poles", "poles_per_oh_km",
    "pole_age_known", "pole_age_mean", "pole_old_share", "pole_wood_share", "pole_height_mean",
    "transformers", "oh_transformer_share", "depth", "history_damage",
]
XF_X = [
    "xf_customers", "xf_overhead", "xf_kva", "xf_single_phase", "secondary_oh_km_per_xf",
    "customers_per_km", "pole_age_mean", "pole_old_share", "pole_wood_share", "pole_age_known", "history_damage",
]
ACCESS_X = [
    "customers_per_km", "primary_overhead_km", "primary_underground_km", "secondary_overhead_km",
    "poles_per_oh_km", "pole_age_mean", "pole_wood_share", "depth",
] + [f"is_{t}" for t in DAMAGE_TYPES]
# skewed counts and lengths get log1p before scaling
LOG = {"customers", "primary_overhead_km", "primary_underground_km", "secondary_overhead_km",
       "secondary_underground_km", "customers_per_km", "poles", "poles_per_oh_km", "transformers",
       "history_damage", "xf_customers", "xf_kva"}


def pipe(cols, model):
    log = [c for c in cols if c in LOG]
    pre = ColumnTransformer([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"), log)],
                            remainder="passthrough", verbose_feature_names_out=False)
    return make_pipeline(pre, StandardScaler(), model).set_output(transform="pandas")


def auc(y, p):
    return {"auc": round(roc_auc_score(y, p), 3), "avg_precision": round(average_precision_score(y, p), 3),
            "base_rate": round(float(np.mean(y)), 4), "mean_pred": round(float(np.mean(p)), 4), "n": int(len(y))}


def home_risk(p_link, hl):
    """1 - prod(1 - p) over each home's links. p_link: Series indexed by link id."""
    # 246 fuses share a location id with a transformer; one damage report labels both, so keep the higher risk
    p_link = p_link.groupby(level=0).max()
    log_safe = np.log1p(-hl.link.map(p_link).fillna(0).clip(upper=0.999))
    return 1 - np.exp(log_safe.groupby(hl.premise_id).sum())


def main():
    zones = pd.read_parquet(FEATURES + "zone_features.parquet")
    xfs = pd.read_parquet(FEATURES + "transformer_features.parquet")
    acc = pd.read_parquet(FEATURES + "access_features.parquet")
    links = pd.read_parquet(FEATURES + "home_links.parquet")
    models, metrics = {}, {}

    # 1. link damaged
    lr = lambda: LogisticRegression(C=1.0, max_iter=2000)
    models["zone_risk"] = pipe(ZONE_X, lr()).fit(zones[ZONE_X], zones[f"damaged_{TRAIN}"])
    models["xf_risk"] = pipe(XF_X, lr()).fit(xfs[XF_X], xfs[f"damaged_{TRAIN}"])
    p_zone = pd.Series(models["zone_risk"].predict_proba(zones[ZONE_X])[:, 1], zones.index)
    p_xf = pd.Series(models["xf_risk"].predict_proba(xfs[XF_X])[:, 1], xfs.index)
    for name, tbl, p in [("zone_risk", zones, p_zone), ("xf_risk", xfs, p_xf)]:
        metrics[name] = {ev: auc(tbl[f"damaged_{ev}"], p) for ev in (TRAIN, TEST)}

    labels = pd.read_parquet(LABELS).set_index("premise_id")
    labels = labels[labels.has_chain]
    p_home = home_risk(pd.concat([p_zone, p_xf]), links).reindex(labels.index).fillna(0)
    metrics["home_risk"] = {ev: auc(labels[f"out_{ev}"], p_home) for ev in (TRAIN, TEST)}

    # 2. damage counts per zone
    models["damage"], metrics["damage"] = {}, {}
    for t in DAMAGE_TYPES:
        m = pipe(ZONE_X, PoissonRegressor(alpha=1e-3, max_iter=2000)).fit(zones[ZONE_X], zones[f"n_{t}_{TRAIN}"])
        models["damage"][t] = m
        pred = m.predict(zones[ZONE_X])
        metrics["damage"][t] = {
            ev: {"spearman": round(pd.Series(pred).corr(pd.Series(zones[f"n_{t}_{ev}"].values), method="spearman"), 3),
                 "predicted_total": round(float(pred.sum()), 1), "actual_total": int(zones[f"n_{t}_{ev}"].sum())}
            for ev in (TRAIN, TEST)}

    # 3. access issue per damage report
    tr, te = acc[acc.event == TRAIN], acc[acc.event == TEST]
    models["access"] = pipe(ACCESS_X, lr()).fit(tr[ACCESS_X], tr.access_issue)
    metrics["access"] = {ev: auc(d.access_issue, models["access"].predict_proba(d[ACCESS_X])[:, 1])
                         for ev, d in ((TRAIN, tr), (TEST, te))}

    os.makedirs(MODELS, exist_ok=True)
    joblib.dump({"models": models, "features": {"zone": ZONE_X, "xf": XF_X, "access": ACCESS_X}}, MODELS + "models.joblib")
    with open(MODELS + "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()


{
  "zone_risk": {
    "feb2025": {
      "auc": 0.836,
      "avg_precision": 0.308,
      "base_rate": 0.0639,
      "mean_pred": 0.0639,
      "n": 6664
    },
    "sep2024": {
      "auc": 0.751,
      "avg_precision": 0.043,
      "base_rate": 0.0131,
      "mean_pred": 0.0639,
      "n": 6664
    }
  },
  "xf_risk": {
    "feb2025": {
      "auc": 0.69,
      "avg_precision": 0.012,
      "base_rate": 0.0056,
      "mean_pred": 0.0056,
      "n": 25303
    },
    "sep2024": {
      "auc": 0.593,
      "avg_precision": 0.004,
      "base_rate": 0.0014,
      "mean_pred": 0.0056,
      "n": 25303
    }
  },
  "home_risk": {
    "feb2025": {
      "auc": 0.802,
      "avg_precision": 0.541,
      "base_rate": 0.1893,
      "mean_pred": 0.2134,
      "n": 117595
    },
    "sep2024": {
      "auc": 0.696,
      "avg_precision": 0.068,
      "base_rate": 0.0368,
      "mean_pred": 0.2134,
      "n": 117595
    }
  },
  "damage": {
    "pole": {
      "feb2025": {
        "spearman": 0